# Django, QuerySet API

## Introduction to QuerySets
---

In this lesson, you'll learn how to **query data** from your database using Django's QuerySet API.

QuerySets are Django's way of reading data - they translate Python code into SQL queries.

1. [QuerySet Basics](#QuerySet-Basics),
    - [The Manager Object](#The-Manager-Object),
    - [What is a QuerySet](#What-is-a-QuerySet),
2. [Retrieving Objects](#Retrieving-Objects),
    - [all() - Get Everything](#all()---Get-Everything),
    - [get() - Get a Single Object](#get()---Get-a-Single-Object),
    - [first() and last()](#first()-and-last()),
3. [Filtering Data](#Filtering-Data),
    - [filter() - Include Matching](#filter()---Include-Matching),
    - [exclude() - Exclude Matching](#exclude()---Exclude-Matching),
    - [Combining Filters](#Combining-Filters),
4. [Field Lookups](#Field-Lookups),
    - [Comparison Lookups](#Comparison-Lookups),
    - [String Lookups](#String-Lookups),
    - [Date Lookups](#Date-Lookups),
    - [Relationship Lookups](#Relationship-Lookups),
5. [QuerySet Chaining and Lazy Evaluation](#QuerySet-Chaining-and-Lazy-Evaluation),
    - [Chaining Methods](#Chaining-Methods),
    - [When QuerySets are Evaluated](#When-QuerySets-are-Evaluated),
6. [Ordering and Limiting](#Ordering-and-Limiting),
    - [order_by()](#order_by()),
    - [Slicing QuerySets](#Slicing-QuerySets),
7. [🧠 EXERCISE 🧠, Query the Bookstore](#🧠-EXERCISE-🧠,-Query-the-Bookstore).

<br>

## QuerySet Basics

---

### The Manager Object

---

Every Django model has a **Manager** - an interface for database operations.

By default, Django adds a manager called `objects` to every model:

<br>

```python
class Book(models.Model):
    title = models.CharField(max_length=200)
    # Django automatically adds: objects = models.Manager()

# Access the manager via .objects
Book.objects  # This is the manager
```

<br>

The manager is your gateway to the database - you use it to:
- **Create** objects: `Book.objects.create(...)`
- **Read** objects: `Book.objects.all()`, `Book.objects.filter(...)`
- **Update** objects: `Book.objects.filter(...).update(...)`
- **Delete** objects: `Book.objects.filter(...).delete()`

<br>

### What is a QuerySet

---

A **QuerySet** represents a collection of objects from your database.

Think of it as a **lazy SQL query** - it describes what you want, but doesn't fetch data until you actually need it.

<br>

| Concept | Description |
|---------|-------------|
| **QuerySet** | A collection of database objects |
| **Lazy** | Query isn't executed until data is needed |
| **Chainable** | Methods return new QuerySets |
| **Iterable** | Can loop over results |

In [ ]:
# Example: QuerySet basics (conceptual - requires Django setup)

# This creates a QuerySet but doesn't hit the database yet
books = Book.objects.all()
# Type: <QuerySet [...]>

# The database is queried when you:
# 1. Iterate over the QuerySet
for book in books:
    print(book.title)

# 2. Slice the QuerySet
first_five = books[:5]

# 3. Convert to list
book_list = list(books)

# 4. Check existence
if books:
    print("We have books!")

<br>

## Retrieving Objects

---

### all() - Get Everything

---

The `all()` method returns a QuerySet containing all objects:

```python
# Get all books
all_books = Book.objects.all()
```

**SQL equivalent:**
```sql
SELECT * FROM catalog_book;
```

In [ ]:
# Example: Using all() in the Django shell

# Get all books
books = Book.objects.all()
print(f"Total books: {books.count()}")

# Iterate over all books
for book in books:
    print(f"- {book.title} by {book.author}")

<br>

### get() - Get a Single Object

---

The `get()` method retrieves **exactly one** object matching the criteria:

```python
# Get book by primary key
book = Book.objects.get(pk=1)

# Get book by unique field
book = Book.objects.get(isbn='978-0134685991')
```

**SQL equivalent:**
```sql
SELECT * FROM catalog_book WHERE id = 1;
```

<br>

**⚠️ Important:** `get()` raises exceptions if:

| Exception | When |
|-----------|------|
| `DoesNotExist` | No object matches |
| `MultipleObjectsReturned` | More than one object matches |

In [ ]:
# Example: Handling get() exceptions

from django.core.exceptions import ObjectDoesNotExist

# Method 1: Try/except
try:
    book = Book.objects.get(pk=999)
except Book.DoesNotExist:
    print("Book not found!")

# Method 2: Use get_or_create (creates if not found)
book, created = Book.objects.get_or_create(
    isbn='978-0134685991',
    defaults={'title': 'Effective Python', 'author': 'Brett Slatkin'}
)
if created:
    print("New book created!")
else:
    print("Book already existed.")

<br>

### first() and last()

---

Use `first()` and `last()` when you want one object but don't want exceptions:

```python
# Returns the first object, or None if empty
book = Book.objects.first()

# Returns the last object, or None if empty
book = Book.objects.last()

# Can be combined with filter
cheapest = Book.objects.order_by('price').first()
```

In [ ]:
# Example: Using first() safely

# Get the newest book (no exception if empty)
newest_book = Book.objects.order_by('-publication_date').first()

if newest_book:
    print(f"Newest: {newest_book.title}")
else:
    print("No books in database.")

<br>

## Filtering Data

---

### filter() - Include Matching

---

The `filter()` method returns a QuerySet with objects matching the given conditions:

```python
# Books by a specific author
python_books = Book.objects.filter(author='Guido van Rossum')

# Books published in 2024
recent_books = Book.objects.filter(publication_year=2024)
```

**SQL equivalent:**
```sql
SELECT * FROM catalog_book WHERE author = 'Guido van Rossum';
```

In [ ]:
# Example: Multiple filter conditions (AND)

# Both conditions must be true (AND)
cheap_python = Book.objects.filter(
    author='Guido van Rossum',
    price__lt=30  # price less than 30
)

# Same as:
cheap_python = Book.objects.filter(author='Guido van Rossum').filter(price__lt=30)

<br>

### exclude() - Exclude Matching

---

The `exclude()` method returns objects that **don't** match the conditions:

```python
# All books except those out of print
available_books = Book.objects.exclude(status='out_of_print')

# Books not by a specific author
other_authors = Book.objects.exclude(author='Unknown')
```

**SQL equivalent:**
```sql
SELECT * FROM catalog_book WHERE NOT status = 'out_of_print';
```

In [ ]:
# Example: Combining filter and exclude

# Published Python books that aren't out of print
available_python_books = (
    Book.objects
    .filter(title__icontains='python')  # Title contains 'python'
    .filter(status='published')          # Status is published
    .exclude(stock=0)                    # Not out of stock
)

<br>

### Combining Filters

---

For **OR** conditions, use the `Q` object:

```python
from django.db.models import Q

# Books by author A OR author B
books = Book.objects.filter(
    Q(author='Author A') | Q(author='Author B')
)

# Complex conditions
books = Book.objects.filter(
    Q(price__lt=20) | Q(on_sale=True),  # Cheap OR on sale
    status='published'  # AND published
)
```

In [ ]:
# Example: Q objects for complex queries

from django.db.models import Q

# Find books that are:
# (cheap AND available) OR (on_sale)
deals = Book.objects.filter(
    (Q(price__lt=15) & Q(stock__gt=0)) | Q(on_sale=True)
)

# Q object operators:
# | = OR
# & = AND
# ~ = NOT

# Books NOT by unknown authors
known_authors = Book.objects.filter(~Q(author='Unknown'))

<br>

## Field Lookups

---

**Field lookups** are how you specify conditions in filters. They use double-underscore syntax: `field__lookup=value`

### Comparison Lookups

---

| Lookup | Description | Example |
|--------|-------------|--------|
| `exact` | Exact match (default) | `name__exact='Python'` |
| `iexact` | Case-insensitive exact | `name__iexact='python'` |
| `gt` | Greater than | `price__gt=20` |
| `gte` | Greater than or equal | `price__gte=20` |
| `lt` | Less than | `price__lt=20` |
| `lte` | Less than or equal | `price__lte=20` |
| `in` | In a list | `status__in=['draft', 'review']` |
| `range` | Between values | `price__range=(10, 50)` |

In [ ]:
# Example: Comparison lookups

# Books costing more than $25
expensive = Book.objects.filter(price__gt=25)

# Books with 100-500 pages
medium_books = Book.objects.filter(page_count__range=(100, 500))

# Books that are draft or under review
pending = Book.objects.filter(status__in=['draft', 'review'])

# Books published in 2024 or later
recent = Book.objects.filter(publication_year__gte=2024)

<br>

### String Lookups

---

| Lookup | Description | Example |
|--------|-------------|--------|
| `contains` | Contains substring | `title__contains='Python'` |
| `icontains` | Case-insensitive contains | `title__icontains='python'` |
| `startswith` | Starts with | `title__startswith='The'` |
| `istartswith` | Case-insensitive starts with | `title__istartswith='the'` |
| `endswith` | Ends with | `title__endswith='Guide'` |
| `iendswith` | Case-insensitive ends with | `title__iendswith='guide'` |
| `regex` | Regular expression | `title__regex=r'^[A-Z]'` |
| `iregex` | Case-insensitive regex | `title__iregex=r'^the'` |

In [ ]:
# Example: String lookups

# Books with 'django' in title (case-insensitive)
django_books = Book.objects.filter(title__icontains='django')
# Matches: "Django for Beginners", "Two Scoops of Django", "DJANGO MASTER"

# Books starting with "The"
the_books = Book.objects.filter(title__startswith='The')
# Matches: "The Pragmatic Programmer", "The Python Handbook"

# Email addresses from gmail
gmail_users = User.objects.filter(email__iendswith='@gmail.com')

<br>

### Date Lookups

---

| Lookup | Description | Example |
|--------|-------------|--------|
| `date` | Exact date | `created_at__date=date(2024, 3, 15)` |
| `year` | Year component | `publication_date__year=2024` |
| `month` | Month component | `publication_date__month=12` |
| `day` | Day component | `publication_date__day=25` |
| `week` | Week number | `created_at__week=1` |
| `week_day` | Day of week (1=Sunday) | `created_at__week_day=2` |

In [ ]:
# Example: Date lookups

from datetime import date, timedelta

# Books published in 2024
books_2024 = Book.objects.filter(publication_date__year=2024)

# Books published in December
december_books = Book.objects.filter(publication_date__month=12)

# Books published in the last 30 days
thirty_days_ago = date.today() - timedelta(days=30)
recent_books = Book.objects.filter(publication_date__gte=thirty_days_ago)

# Orders created today
from django.utils import timezone
today_orders = Order.objects.filter(created_at__date=timezone.now().date())

<br>

### Relationship Lookups

---

You can traverse relationships using double underscores:

```python
# Books by authors named "John"
books = Book.objects.filter(author__first_name='John')

# Books in the "Science Fiction" category
books = Book.objects.filter(categories__name='Science Fiction')
```

**SQL equivalent:**
```sql
SELECT * FROM catalog_book 
JOIN catalog_author ON book.author_id = author.id
WHERE author.first_name = 'John';
```

In [ ]:
# Example: Spanning relationships

# Books by authors from USA
american_books = Book.objects.filter(author__country='USA')

# Books in categories containing 'fiction' in name
fiction_books = Book.objects.filter(categories__name__icontains='fiction')

# Orders containing books by a specific author
orders = Order.objects.filter(items__book__author__name='William Vincent')

# Authors who have published books (reverse relationship)
published_authors = Author.objects.filter(books__status='published').distinct()

<br>

| Lookup | Description |
|--------|-------------|
| `isnull` | Check for NULL | `author__isnull=True` |
| Spanning FK | Access related field | `author__name='John'` |
| Spanning M2M | Access many-to-many | `categories__name='Sci-Fi'` |

<br>

## QuerySet Chaining and Lazy Evaluation

---

### Chaining Methods

---

Most QuerySet methods return a **new QuerySet**, allowing you to chain them:

```python
# Build up a complex query step by step
books = (
    Book.objects
    .filter(status='published')
    .filter(price__lt=30)
    .exclude(stock=0)
    .order_by('-publication_date')
    [:10]  # First 10 results
)
```

Each method creates a new QuerySet without modifying the original!

In [ ]:
# Example: Building queries dynamically

def search_books(title=None, author=None, max_price=None, category=None):
    """Search books with optional filters."""
    queryset = Book.objects.filter(status='published')
    
    if title:
        queryset = queryset.filter(title__icontains=title)
    
    if author:
        queryset = queryset.filter(author__name__icontains=author)
    
    if max_price:
        queryset = queryset.filter(price__lte=max_price)
    
    if category:
        queryset = queryset.filter(categories__name=category)
    
    return queryset.order_by('title')

# Usage:
# results = search_books(title='python', max_price=50)

<br>

### When QuerySets are Evaluated

---

QuerySets are **lazy** - they don't hit the database until you actually need the data.

<br>

**QuerySets are evaluated when you:**

| Action | Example |
|--------|--------|
| Iterate | `for book in books:` |
| Slice with step | `books[::2]` |
| Convert to list | `list(books)` |
| Check bool | `if books:` |
| Call `len()` | `len(books)` |
| Call `repr()` | `print(books)` |
| Pickle | `pickle.dumps(books)` |

In [ ]:
# Example: Understanding lazy evaluation

# This does NOT hit the database
books = Book.objects.filter(status='published')
books = books.filter(price__lt=30)
books = books.order_by('title')
# Still no database query!

# NOW the database is queried (iteration)
for book in books:
    print(book.title)

# Benefit: You can build complex queries without performance penalty
# Only ONE SQL query is executed, no matter how many filter() calls

<br>

**Caching:** Once evaluated, results are cached in the QuerySet:

```python
# First iteration - database query
for book in books:
    print(book.title)

# Second iteration - uses cached results, no database query
for book in books:
    print(book.author)
```

<br>

## Ordering and Limiting

---

### order_by()

---

The `order_by()` method sorts results:

```python
# Ascending (default)
books = Book.objects.order_by('title')

# Descending (prefix with -)
books = Book.objects.order_by('-publication_date')

# Multiple fields
books = Book.objects.order_by('author', '-publication_date')
# First by author (A-Z), then by date (newest first)
```

**SQL equivalent:**
```sql
SELECT * FROM catalog_book ORDER BY title ASC;
SELECT * FROM catalog_book ORDER BY publication_date DESC;
```

In [ ]:
# Example: Various ordering options

# Alphabetical by title
alphabetical = Book.objects.order_by('title')

# Price low to high
cheapest_first = Book.objects.order_by('price')

# Price high to low
expensive_first = Book.objects.order_by('-price')

# By related field
by_author = Book.objects.order_by('author__last_name', 'author__first_name')

# Random order
random_books = Book.objects.order_by('?')  # Note: Can be slow on large tables!

# Clear existing ordering (from Meta class)
unordered = Book.objects.order_by()

<br>

### Slicing QuerySets

---

You can slice QuerySets like Python lists to limit results:

```python
# First 5 books
first_five = Book.objects.all()[:5]

# Books 5-10 (offset 5, limit 5)
next_five = Book.objects.all()[5:10]

# Single item (returns object, not QuerySet)
sixth_book = Book.objects.all()[5]
```

**SQL equivalent:**
```sql
SELECT * FROM catalog_book LIMIT 5;
SELECT * FROM catalog_book LIMIT 5 OFFSET 5;
```

In [ ]:
# Example: Pagination with slicing

def get_page(page_number: int, page_size: int = 10):
    """Get a page of books."""
    start = (page_number - 1) * page_size
    end = start + page_size
    return Book.objects.order_by('title')[start:end]

# Usage:
# page_1 = get_page(1)  # Books 0-9
# page_2 = get_page(2)  # Books 10-19
# page_3 = get_page(3)  # Books 20-29

<br>

**Note:** Negative indexing is NOT supported:

```python
# This will raise an error!
last_book = Book.objects.all()[-1]  # AssertionError

# Use this instead:
last_book = Book.objects.order_by('-id').first()
# or
last_book = Book.objects.last()
```

<br>

## Useful QuerySet Methods

---

Here are some additional methods you'll use frequently:

In [ ]:
# count() - Get the count without loading all objects
book_count = Book.objects.filter(status='published').count()
# SQL: SELECT COUNT(*) FROM catalog_book WHERE status = 'published';

# exists() - Check if any objects match (efficient)
has_books = Book.objects.filter(author__name='Unknown').exists()
# Returns True/False without loading objects

# values() - Get dictionaries instead of model instances
titles = Book.objects.values('title', 'author__name')
# Returns: [{'title': 'Book 1', 'author__name': 'John'}, ...]

# values_list() - Get tuples instead of dictionaries
titles = Book.objects.values_list('title', flat=True)
# Returns: ['Book 1', 'Book 2', 'Book 3', ...]

# distinct() - Remove duplicates
unique_authors = Book.objects.values('author').distinct()

<br>

## Summary

---

| Method | Description |
|--------|-------------|
| `all()` | Get all objects |
| `get()` | Get single object (raises exception if not found) |
| `first()` / `last()` | Get first/last object (or None) |
| `filter()` | Include objects matching conditions |
| `exclude()` | Exclude objects matching conditions |
| `order_by()` | Sort results |
| `count()` | Count objects efficiently |
| `exists()` | Check if any objects exist |
| `values()` | Return dictionaries instead of objects |

<br>

**Common lookups:**

| Lookup | Example |
|--------|--------|
| `__exact` | `name__exact='value'` (default) |
| `__icontains` | `title__icontains='python'` |
| `__gt`, `__lt` | `price__gt=20`, `price__lt=50` |
| `__in` | `status__in=['draft', 'review']` |
| `__isnull` | `author__isnull=True` |
| `__year`, `__month` | `date__year=2024` |

<br>

### 🧠 EXERCISE 🧠, Query the Bookstore

---

Using the Django shell, practice QuerySet operations:

1. Get all published books
2. Get the 5 most expensive books
3. Find books with "Python" in the title (case-insensitive)
4. Get books published in the last year
5. Find books by authors whose name contains "Vincent"
6. Count how many books are in stock (stock > 0)
7. Get a list of unique author names (use `values_list`)
8. Find books that are either under $20 OR have more than 100 in stock

<details>
    <summary>▶️ Solution</summary>
    
```python
# Start Django shell
# python manage.py shell

from catalog.models import Book
from django.db.models import Q
from datetime import date, timedelta

# 1. All published books
published = Book.objects.filter(status='published')

# 2. 5 most expensive books
expensive = Book.objects.order_by('-price')[:5]

# 3. Books with "Python" in title
python_books = Book.objects.filter(title__icontains='python')

# 4. Books published in last year
one_year_ago = date.today() - timedelta(days=365)
recent = Book.objects.filter(publication_date__gte=one_year_ago)

# 5. Books by authors containing "Vincent"
vincent_books = Book.objects.filter(author__name__icontains='vincent')

# 6. Count books in stock
in_stock_count = Book.objects.filter(stock__gt=0).count()

# 7. Unique author names
authors = Book.objects.values_list('author__name', flat=True).distinct()

# 8. Cheap OR high stock
deals = Book.objects.filter(
    Q(price__lt=20) | Q(stock__gt=100)
)
```
</details>

<br>

### 🧠 EXERCISE 🧠, Build a Search Function

---

Create a search function for books that accepts optional parameters:

1. `query` - search in title and description (case-insensitive)
2. `min_price` - minimum price
3. `max_price` - maximum price
4. `category` - filter by category name
5. `in_stock_only` - only show books with stock > 0

The function should:
- Only apply filters for provided parameters
- Always filter to published books only
- Order results by relevance (title match first) then by publication date

<details>
    <summary>▶️ Solution</summary>
    
```python
from django.db.models import Q, Case, When, Value, IntegerField

def search_books(
    query: str = None,
    min_price: float = None,
    max_price: float = None,
    category: str = None,
    in_stock_only: bool = False
):
    """Search books with optional filters."""
    # Start with published books
    qs = Book.objects.filter(status='published')
    
    # Search query in title and description
    if query:
        qs = qs.filter(
            Q(title__icontains=query) | Q(description__icontains=query)
        )
    
    # Price range
    if min_price is not None:
        qs = qs.filter(price__gte=min_price)
    if max_price is not None:
        qs = qs.filter(price__lte=max_price)
    
    # Category filter
    if category:
        qs = qs.filter(categories__name__iexact=category)
    
    # In stock filter
    if in_stock_only:
        qs = qs.filter(stock__gt=0)
    
    # Order by title match (if query) then by date
    if query:
        qs = qs.annotate(
            title_match=Case(
                When(title__icontains=query, then=Value(0)),
                default=Value(1),
                output_field=IntegerField()
            )
        ).order_by('title_match', '-publication_date')
    else:
        qs = qs.order_by('-publication_date')
    
    return qs

# Usage:
# results = search_books(query='django', max_price=50, in_stock_only=True)
```
</details>

---